# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data in record sets, each with unique `@id`s and associated fields. We list the record sets and their fields for exploration.

In [ ]:
# Fetch record sets and fields from Croissant schema
# All entities are referenced by their @id fields
record_sets = []

# The metadata.record_set attribute should be a list of RecordSet objects
if hasattr(metadata, 'record_set') and metadata.record_set:
    for rset in metadata.record_set:
        print(f"Record Set @id: {rset['@id']} | Name: {rset.get('name', 'N/A')}")
        record_sets.append(rset['@id'])
        # Fields for the record set
        fields = rset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"\tField @id: {field['@id']} | Name: {field.get('name', 'N/A')} | DataType: {field.get('dataType', 'N/A')}")
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Use the list of record set @id's obtained above
dataframes = {}

# If no record sets detected, fallback to Croissant default record set @id for the FAIR² dataset
if not record_sets:
    record_sets = ['https://api.app.sen.science/frontiers/7160411/mental_health_survey']  # Replace with actual @id from schema if known
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded dataframe for Record Set @id: {record_set_id}")
    print("Columns:", dataframes[record_set_id].columns.tolist())
    print(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll reference fields by their `@id` (Examples: 'cr:GAD7_score', 'cr:age', 'cr:village', etc. – your actual field @id's may differ, check the output above).

In [ ]:
# Choose a record set and numeric field for EDA
# Replace the field and group by @id's with appropriate values from your data overview
if record_sets:
    main_record_set_id = record_sets[0]
else:
    main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/mental_health_survey'  # fallback

# Example field @id (update as appropriate):
numeric_field_id = 'cr:GAD7_score' # Example: GAD-7 Score. Replace with correct @id.

# Explore available columns
df = dataframes[main_record_set_id]
print("Available columns:", df.columns.tolist())

if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Example group field @id for grouping: 'cr:village' (replace with actual)
    group_field_id = 'cr:village'
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We use matplotlib to plot the distribution of the GAD-7 score and its mean per village, referencing fields by their @id.

In [ ]:
# Visualization example: Distribution and grouping
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the GAD7 score (cr:GAD7_score)
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score (@id: cr:GAD7_score)')
    plt.ylabel('Frequency')
    plt.show()
    
    # Plot group means if grouping occurred above
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 6))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'Mean GAD-7 Score by Village (@id: {group_field_id})')
        plt.xlabel(f'Village (@id: {group_field_id})')
        plt.ylabel(f'Mean GAD-7 Score (@id: {numeric_field_id})')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using `mlcroissant`, referencing entities by their `@id`.
- Loaded and reviewed available record sets and fields, demonstrating extraction to pandas DataFrames.
- Performed EDA by filtering and normalizing scores, and grouping by village.
- Visualized distributions of key psychological indicators.
- This AI-ready, FAIR-structured dataset enables advanced analysis, supporting mental health research and community interventions in Africa.